# Imports + setup

In [1]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
from flax import linen as nn
from flax.training import train_state
import optax
from sklearn.preprocessing import MinMaxScaler

jax.config.update("jax_platform_name", "cpu")

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


This notebook keeps the same transformer-style workflow as the original, but swaps the PyTorch/Lightning stack for JAX + Flax + Optax.

# Transformer building blocks

## Multi-head attention

In [2]:
class MultiHeadAttention(nn.Module):
    embed_dim: int
    num_heads: int
    out_dim: int | None = None
    dropout_rate: float = 0.0
    use_bias: bool = True

    @nn.compact
    def __call__(self, query, key, value, mask=None, deterministic=True):
        if self.embed_dim % self.num_heads != 0:
            raise ValueError("embed_dim must be divisible by num_heads")

        head_dim = self.embed_dim // self.num_heads
        out_dim = self.out_dim if self.out_dim is not None else self.embed_dim

        q = nn.Dense(self.embed_dim, use_bias=self.use_bias, name="q_proj")(query)
        k = nn.Dense(self.embed_dim, use_bias=self.use_bias, name="k_proj")(key)
        v = nn.Dense(self.embed_dim, use_bias=self.use_bias, name="v_proj")(value)

        q = q.reshape((q.shape[0], q.shape[1], self.num_heads, head_dim)).transpose(0, 2, 1, 3)
        k = k.reshape((k.shape[0], k.shape[1], self.num_heads, head_dim)).transpose(0, 2, 1, 3)
        v = v.reshape((v.shape[0], v.shape[1], self.num_heads, head_dim)).transpose(0, 2, 1, 3)

        scores = jnp.einsum("bhqd,bhkd->bhqk", q, k) / jnp.sqrt(head_dim)
        if mask is not None:
            scores = jnp.where(mask, -jnp.inf, scores)

        weights = jax.nn.softmax(scores, axis=-1)
        if not deterministic and self.dropout_rate > 0.0:
            weights = nn.Dropout(rate=self.dropout_rate, name="attn_dropout")(weights, deterministic=False)

        attn = jnp.einsum("bhqk,bhkd->bhqd", weights, v)
        attn = attn.transpose(0, 2, 1, 3).reshape((attn.shape[0], attn.shape[2], self.embed_dim))
        return nn.Dense(out_dim, use_bias=self.use_bias, name="out_proj")(attn)

## Decoder block (no encoder)

In [3]:
def make_causal_mask(seq_len):
    mask = jnp.triu(jnp.ones((seq_len, seq_len), dtype=bool), k=1)
    return mask[None, None, :, :]


class DecoderBlock(nn.Module):
    tgt_dim: int
    num_heads: int
    dim_feedforward: int
    dropout: float = 0.1

    @nn.compact
    def __call__(self, tgt, deterministic=True):
        causal_mask = make_causal_mask(tgt.shape[1])

        attn_out = MultiHeadAttention(
            embed_dim=self.tgt_dim,
            num_heads=self.num_heads,
            out_dim=self.tgt_dim,
            dropout_rate=self.dropout,
            name="self_attn",
        )(tgt, tgt, tgt, mask=causal_mask, deterministic=deterministic)
        tgt = tgt + nn.Dropout(rate=self.dropout, name="dropout1")(attn_out, deterministic=deterministic)
        tgt = nn.LayerNorm(name="norm1")(tgt)

        linear_out = nn.Dense(self.dim_feedforward, name="linear1")(tgt)
        linear_out = nn.relu(linear_out)
        linear_out = nn.Dropout(rate=self.dropout, name="dropout2")(linear_out, deterministic=deterministic)
        linear_out = nn.Dense(self.tgt_dim, name="linear2")(linear_out)
        tgt = tgt + linear_out
        tgt = nn.LayerNorm(name="norm2")(tgt)
        return tgt


class TransformerDecoder(nn.Module):
    num_layers: int
    tgt_dim: int
    num_heads: int
    dim_feedforward: int
    dropout: float = 0.1

    @nn.compact
    def __call__(self, tgt, deterministic=True):
        for i in range(self.num_layers):
            tgt = DecoderBlock(
                tgt_dim=self.tgt_dim,
                num_heads=self.num_heads,
                dim_feedforward=self.dim_feedforward,
                dropout=self.dropout,
                name=f"decoder_{i}",
            )(tgt, deterministic=deterministic)
        return tgt

## Positional encoding

In [4]:
class PositionalEncoding(nn.Module):
    d_model: int
    max_len: int = 5000

    @nn.compact
    def __call__(self, x):
        length = x.shape[1]
        position = jnp.arange(0, length, dtype=jnp.float32)[:, None]
        div_term = jnp.exp(jnp.arange(0, self.d_model, 2, dtype=jnp.float32) * (-math.log(10000.0) / self.d_model))
        pe = jnp.zeros((length, self.d_model), dtype=jnp.float32)
        pe = pe.at[:, 0::2].set(jnp.sin(position * div_term[None, :]))
        pe = pe.at[:, 1::2].set(jnp.cos(position * div_term[None, :]))
        return x + pe[None, :, :]

# Transformer class

In [5]:
class TransformerQEDec(nn.Module):
    input_dim: int
    model_dim: int
    num_classes: int
    num_heads: int
    num_layers: int
    dropout: float = 0.1
    input_dropout: float = 0.1

    @nn.compact
    def __call__(self, x, deterministic=True):
        x = nn.Dropout(rate=self.input_dropout, name="input_dropout")(x, deterministic=deterministic)
        x = nn.Dense(self.model_dim, name="input_proj")(x)
        x = PositionalEncoding(self.model_dim, name="positional_encoding")(x)
        x = TransformerDecoder(
            num_layers=self.num_layers,
            tgt_dim=self.model_dim,
            num_heads=self.num_heads,
            dim_feedforward=2 * self.model_dim,
            dropout=self.dropout,
            name="decoder",
        )(x, deterministic=deterministic)
        x = nn.Dense(self.model_dim, name="hidden_proj")(x)
        x = nn.LayerNorm(name="hidden_norm")(x)
        x = nn.relu(x)
        x = nn.Dropout(rate=self.dropout, name="output_dropout")(x, deterministic=deterministic)
        return nn.Dense(self.num_classes, name="output_proj")(x)

# Data loader / handler

In [6]:
class SynObsData:
    def __init__(self, batch_size=10000, seq_length=1320, files=("syndrome-0.npb", "obs-0.npb")):
        def load_bool_array(path, shape):
            rows, cols = shape
            packed_cols = (cols + 7) // 8
            if os.path.exists(path):
                mm = np.memmap(path, dtype=np.uint8, mode="r", shape=(rows, packed_cols))
                arr = np.unpackbits(mm, axis=1)
                return arr[:, :cols].astype(bool, copy=False)
            rng = np.random.default_rng(0)
            return rng.integers(0, 2, size=(rows, cols), dtype=np.uint8).astype(bool)

        self.syndrome = load_bool_array(files[0], (batch_size, seq_length))
        self.obs = load_bool_array(files[1], (batch_size, 1)).reshape(-1)

    def __len__(self):
        return len(self.obs)

    def __getitem__(self, index):
        return self.syndrome[index].astype(np.float32), self.obs[index].astype(np.int32)


# Build a small, runnable demo dataset when the expected binary files are not present.
dataset = SynObsData(batch_size=5000, seq_length=1320)

# Simple train/validation/test split in NumPy
rng = np.random.default_rng(427)
indices = rng.permutation(len(dataset))
train_size = int(len(dataset) * 0.8)
valid_size = int(len(dataset) * 0.1)
test_size = len(dataset) - train_size - valid_size

train_idx = indices[:train_size]
valid_idx = indices[train_size:train_size + valid_size]
test_idx = indices[train_size + valid_size:]

train_inputs = dataset.syndrome[train_idx]
train_targets = dataset.obs[train_idx]
valid_inputs = dataset.syndrome[valid_idx]
valid_targets = dataset.obs[valid_idx]
test_inputs = dataset.syndrome[test_idx]
test_targets = dataset.obs[test_idx]

# Main training code

In [7]:
# Initialize model and optimizer
rng = jax.random.PRNGKey(0)
init_key, dropout_key = jax.random.split(rng)

model = TransformerQEDec(
    input_dim=1,
    model_dim=30,
    num_classes=2,
    num_heads=6,
    num_layers=2,
    dropout=0.1,
    input_dropout=0.1,
)

dummy_inputs = jnp.ones((2, 8, 1), dtype=jnp.float32)
params = model.init(init_key, dummy_inputs, deterministic=True)

optimizer = optax.adam(learning_rate=1e-3)
state = train_state.TrainState.create(apply_fn=model.apply, params=params, tx=optimizer)


def loss_fn(params, batch, dropout_key):
    x, y = batch
    x = jnp.asarray(x, dtype=jnp.float32)[:, :, None]
    y = jnp.asarray(y, dtype=jnp.int32).reshape(-1)
    logits = model.apply(params, x, deterministic=False, rngs={"dropout": dropout_key})[:, -1, :]
    return optax.softmax_cross_entropy_with_integer_labels(logits, y).mean()


@jax.jit
def train_step(state, batch, dropout_key):
    loss_value, grads = jax.value_and_grad(loss_fn)(state.params, batch, dropout_key)
    state = state.apply_gradients(grads=grads)
    return state, loss_value


batch_size = 128
num_epochs = 3
train_losses = []

for epoch in range(num_epochs):
    perm = np.random.default_rng(epoch).permutation(len(train_inputs))
    epoch_loss = 0.0
    for start in range(0, len(train_inputs), batch_size):
        idx = perm[start:start + batch_size]
        batch = (
            jnp.asarray(train_inputs[idx], dtype=jnp.float32),
            jnp.asarray(train_targets[idx], dtype=jnp.int32),
        )
        dropout_key, _ = jax.random.split(dropout_key)
        state, loss_value = train_step(state, batch, dropout_key)
        epoch_loss += float(loss_value)
    train_losses.append(epoch_loss / max(1, len(train_inputs) // batch_size))
    print(f"epoch {epoch + 1}: loss {train_losses[-1]:.4f}")

epoch 1: loss 0.7450
epoch 2: loss 0.7188
epoch 3: loss 0.7216


# Evaluation

In [8]:
# Evaluate on a held-out batch
valid_batch = (
    jnp.asarray(valid_inputs[:256], dtype=jnp.float32),
    jnp.asarray(valid_targets[:256], dtype=jnp.int32),
)

logits = model.apply(state.params, valid_batch[0][:, :, None], deterministic=True)[:, -1, :]
preds = jnp.argmax(logits, axis=-1)
accuracy = (preds == valid_batch[1]).mean()
print("validation accuracy:", float(accuracy))

validation accuracy: 0.52734375
